# Recap Exercises

In this noteboook we are going to import a dataset into a Hugging Face `Dataset` and `DatasetDict`. Then we shall use the `.filter` and `.map` function to create new features and filter the dataset.  

## Task

Import `Dataset`, `DatasetDict` and `load_dataset`.

In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset, ClassLabel

/Users/jacktovey/Library/CloudStorage/OneDrive-UniversityofChichester/COM407-AppliedAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task

We are going to import the 2025 sold house prices dataset from HM Land Registry found here:

https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads

Load the CSV for 2025 using `load_dataset()`. The `csv` file does not have a column header and instead you need to give the column names of:

`col_names = [
    "Transaction_unique_identifier",
    "Price",
    "Date_of_Transfer",
    "Postcode",
    "Property_Type",
    "Old_New",
    "Duration",
    "PAON",
    "SAON",
    "Street",
    "Locality",
    "Town_City",
    "District",
    "County",
    "PPD_Category_Type",
    "Record_Status"
]`


Once imported, look at the column names and check the first couple of rows.

In [2]:
col_names = [
    "Transaction_unique_identifier",
    "Price",
    "Date_of_Transfer",
    "Postcode",
    "Property_Type",
    "Old_New",
    "Duration",
    "PAON",
    "SAON",
    "Street",
    "Locality",
    "Town_City",
    "District",
    "County",
    "PPD_Category_Type",
    "Record_Status"
]

In [3]:
import requests

url = "https://price-paid-data.publicdata.landregistry.gov.uk/pp-2025.csv"
response = requests.get(url)

if response.status_code == 200:
    with open("pp-2025.csv", "wb") as f:
        f.write(response.content)

    csv_loaded = load_dataset("csv", data_files="pp-2025.csv", column_names=col_names)
else:
    print(f"Failed to download: {response.status_code}")


Generating train split: 757816 examples [00:01, 495761.70 examples/s]


## Task

Property types are:

D = Detached

S = Semi-Detached

T = Terraced

F = Flats/Maisonettes

O = Other

Remove all rows where property type is "O".
Verify the remaining unique property types.


In [4]:
csv_loaded = csv_loaded.filter(lambda x: x["Property_Type"] != "O")

Filter: 100%|██████████| 757816/757816 [00:03<00:00, 210126.74 examples/s]


## Task

Calculate the frequency of the property types in the dataset.

In [5]:
# sample = pd.DataFrame(csv_loaded['train'][:])
# sample

In [6]:
type_values = set(csv_loaded['train']['Property_Type'])
type_values

{'D', 'F', 'S', 'T'}

In [7]:
csv_loaded['train']['Property_Type'].count('T')

220009

In [8]:
freq = {}
for t in type_values:
    freq[t] = csv_loaded['train']['Property_Type'].count(t)
freq

{'T': 220009, 'D': 174353, 'F': 123802, 'S': 214249}

## Task

Create a 60/40 split using a random shuffle `seed=42` and stratify by property type to maintain the property type distribution.

Split the remaining 40% into 20% validation and 20% test and combine into a `DatasetDict` with keys of "train", "validation" and "test".

Verify class proportions in each split.

In [9]:
csv_loaded['train'] = csv_loaded['train'].class_encode_column('Property_Type')

Casting to class labels: 100%|██████████| 732413/732413 [00:00<00:00, 1190398.57 examples/s]


In [10]:
split1 = csv_loaded['train'].train_test_split(test_size=0.4, seed=42, stratify_by_column='Property_Type')
split2 = split1['test'].train_test_split(test_size=0.5, seed=42, stratify_by_column='Property_Type')
dataset_dict = DatasetDict({
    'train': split1['train'],
    'validation': split2['train'],
    'test': split2['test']
})

## Task

Use the `map` function to create a new column which puts properties into the categories of:

- "Budget": Price < 250000
- "Mid": 250000 ≤ Price ≤ 750000
- "Premium": Price > 750000

In [11]:
housedata = dataset_dict.map(lambda x: {"Price_Category": "Budget" if x["Price"] < 250000 else "Mid" if 250000 <= x["Price"] <= 750000 else "Premium"})

Map: 100%|██████████| 146483/146483 [00:07<00:00, 19117.26 examples/s]


## Task

Filter the dataset so that only properties:
- Sold for more than £50,000
- Sold for less than £1,500,000

Apply filtering to all splits and report the number of rows in each split after filtering.

In [12]:
housedata.filter(lambda x: 50000 < x["Price"] < 1500000)

Filter: 100%|██████████| 146483/146483 [00:00<00:00, 211331.02 examples/s]


DatasetDict({
    train: Dataset({
        features: ['Transaction_unique_identifier', 'Price', 'Date_of_Transfer', 'Postcode', 'Property_Type', 'Old_New', 'Duration', 'PAON', 'SAON', 'Street', 'Locality', 'Town_City', 'District', 'County', 'PPD_Category_Type', 'Record_Status', 'Price_Category'],
        num_rows: 432973
    })
    validation: Dataset({
        features: ['Transaction_unique_identifier', 'Price', 'Date_of_Transfer', 'Postcode', 'Property_Type', 'Old_New', 'Duration', 'PAON', 'SAON', 'Street', 'Locality', 'Town_City', 'District', 'County', 'PPD_Category_Type', 'Record_Status', 'Price_Category'],
        num_rows: 144362
    })
    test: Dataset({
        features: ['Transaction_unique_identifier', 'Price', 'Date_of_Transfer', 'Postcode', 'Property_Type', 'Old_New', 'Duration', 'PAON', 'SAON', 'Street', 'Locality', 'Town_City', 'District', 'County', 'PPD_Category_Type', 'Record_Status', 'Price_Category'],
        num_rows: 144339
    })
})

## Task

Save the dataset into the `assets` folder for week 4.

In [13]:
housedata.save_to_disk("//Users/jacktovey/Library/CloudStorage/OneDrive-UniversityofChichester/COM407-AppliedAI/week_4/assets/housedata", max_shard_size="100MB")

Saving the dataset (1/1 shards): 100%|██████████| 146483/146483 [00:00<00:00, 1912886.36 examples/s]
